In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import windows
from ipywidgets import Checkbox, VBox, HBox, HTML, Layout, interactive_output
from IPython.display import display

# Window length parameter
M = 100

# Dictionary containing theoretical properties and descriptions (in English)
window_properties = {
    'Rectangular': {
        'color': '#00bcd4',
        'sidelobe': '-13 dB',
        'main_lobe': r'2 $\pi / M$ (Very narrow)',
        'desc': 'Maximum spectral leakage, abrupt truncation.'
    },
    'Hamming': {
        'color': '#9c27b0',
        'sidelobe': '-43 dB',
        'main_lobe': r'4 $\pi / M$',
        'desc': 'Good compromise, excellent first sidelobe attenuation.'
    },
    'von Hann': {
        'color': '#e91e63',
        'sidelobe': '-31 dB',
        'main_lobe': r'4 $\pi / M$',
        'desc': 'Smooth cosine-bell shape, rapid sidelobe roll-off.'
    },
    'Blackman': {
        'color': '#ffeb3b',
        'sidelobe': '-58 dB',
        'main_lobe': r'6 $\pi / M$ (Wide)',
        'desc': 'Very high sidelobe attenuation at the cost of a wider main lobe.'
    },
    'Kaiser (β=5)': {
        'color': '#f44336',
        'sidelobe': r'Adjustable ($\approx -30$ to $-40$ dB)',
        'main_lobe': 'Flexible',
        'desc': 'Shape parameter beta dynamically controls main-to-sidelobe trade-off.'
    },
    'Bartlett': {
        'color': '#009688',
        'sidelobe': '-26 dB',
        'main_lobe': r'4 $\pi / M$',
        'desc': 'Triangular shape, simple implementation with moderate leakage.'
    }
}

def plot_and_show_info_clean(rect, hamm, hann, black, kais, bart):
    plt.close('all')
    
    fig, ax = plt.subplots(figsize=(10, 5))
    n_fft = 2048
    
    active_wins = {
        'Rectangular': (rect, windows.boxcar(M)),
        'Hamming':     (hamm, windows.hamming(M)),
        'von Hann':    (hann, windows.hann(M)),
        'Blackman':    (black, windows.blackman(M)),
        'Kaiser (β=5)':(kais, windows.kaiser(M, 5.0)),
        'Bartlett':    (bart, windows.bartlett(M))
    }
    
    any_active = False
    html_info_content = ""
    
    for name, (is_active, w) in active_wins.items():
        color = window_properties[name]['color']
        if is_active:
            any_active = True
            W = np.fft.rfft(w, n_fft)
            mag = np.abs(W)
            mag_db = 20 * np.log10(mag / np.max(mag) + 1e-12)
            omega = np.linspace(0, np.pi, len(mag_db))
            
            ax.plot(omega / np.pi, mag_db, label=name, color=color, lw=1.5)
            
            prop = window_properties[name]
            html_info_content += f"""
            <div style="margin: 0px; padding: 3px 0px; border-left: 3px solid {color}; padding-left: 8px; font-family: sans-serif; font-size: 12px; line-height: 1.3;">
                <b>{name}:</b> 
                <span><b>First Sidelobe:</b> {prop['sidelobe']} | </span>
                <span><b>Main Lobe:</b> {prop['main_lobe']} | </span>
                <span style="color: #555; font-style: italic;">{prop['desc']}</span>
            </div>
            """
            
    ax.set_title('Comparison of Window Spectral Responses (Magnitude in dB)', fontsize=11, fontweight='bold')
    ax.set_xlabel(r'Normalized Frequency ($\times \pi$ rad/sample)', fontsize=9)
    ax.set_ylabel(r'Normalized Magnitude (dB)', fontsize=9)
    ax.set_xlim(0, 1)
    ax.set_ylim(-100, 5)
    ax.grid(True, which='both', linestyle='--', alpha=0.6)
    
    if any_active:
        ax.legend(loc='upper right', fontsize=8)
    else:
        ax.text(0.5, -50, 'No window selected.', 
                horizontalalignment='center', verticalalignment='center', 
                fontsize=10, color='#666666', bbox=dict(boxstyle='round,pad=0.5', facecolor='#f8f9fa'))
        html_info_content = "<i style='font-family: sans-serif; font-size: 12px; color: #666;'>Please select at least one window to view its properties.</i>"
        
    plt.tight_layout()
    plt.show()
    
    info_box.value = html_info_content

# Checkboxes with tightly compressed horizontal spacing
cb_rect  = Checkbox(value=True, description='Rectangular', layout=Layout(width='auto', margin='0px -12px 0px 0px'))
cb_hamm  = Checkbox(value=True, description='Hamming', layout=Layout(width='auto', margin='0px -12px 0px 0px'))
cb_hann  = Checkbox(value=True, description='von Hann', layout=Layout(width='auto', margin='0px -12px 0px 0px'))
cb_black = Checkbox(value=True, description='Blackman', layout=Layout(width='auto', margin='0px -12px 0px 0px'))
cb_kais  = Checkbox(value=True, description='Kaiser (β=5)', layout=Layout(width='auto', margin='0px -12px 0px 0px'))
cb_bart  = Checkbox(value=True, description='Bartlett', layout=Layout(width='auto', margin='0px -12px 0px 0px'))

info_box = HTML()

interactive_plot = interactive_output(
    plot_and_show_info_clean, 
    {
        'rect': cb_rect, 'hamm': cb_hamm, 'hann': cb_hann, 
        'black': cb_black, 'kais': cb_kais, 'bart': cb_bart
    }
)

# Horizontal layout for checkboxes to sit nicely between the plot and info box
checkbox_bar = HBox([cb_rect, cb_hamm, cb_hann, cb_black, cb_kais, cb_bart], layout=Layout(margin='5px 0px 10px 0px'))

# Final dashboard layout stacked vertically: Plot -> Checkboxes -> Properties Info
dashboard = VBox([
    interactive_plot,
    checkbox_bar,
    info_box
])

display(dashboard)